In [1]:
from tqdm.notebook import tqdm
import polars as pl

# wags-llm imports
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner

# dgiLIT Pydantic Models
from dgilit_models import InteractionExtractionResult
from dgilit_models import InteractionExtractionPrompt

# wags-llm
Wags-LLM exists to add structure to LLM prompt calls in a repeatable, explainable way. This is the first attempt at implementing this with a dgiLIT prompt. Other portions of the dgiLIT pipeline would happen before this step and as such, the supporting models and prompts in this branch will likely evolve as those are tuned up.  
  
Notable things to update:  
- pre-tagging of drugs and genes to submit alongside a prompt. 
- support for multiple interactions per one abstract or block of text
- support for chunking sections of a PMID (Abstract, Results, Conclusion)
- support for Pydantic model level sanity checks to reduce LLM usage where not needed (may involve pre-tagging or deterministic rule sets that would provide a `run_llm? TRUE FALSE` type evaluation)

In [ ]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 1000

def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM interaction extraction task runner

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(InteractionExtractionPrompt(version="v1"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

In [3]:
def extraction_interactions(
    task_runner: StructuredTaskRunner,
    prompt: InteractionExtractionPrompt,
    context: str,
    candidate_drugs: list[str] | None = None,
    candidate_genes: list[str] | None = None,
) -> InteractionExtractionResult:
    """Extract zero, one, or many drug-gene interactions from biomedical text."""

    payload = prompt.build_payload(
        context=context,
        candidate_drugs=candidate_drugs or [],
        candidate_genes=candidate_genes or [],
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=InteractionExtractionResult,
        )

        return task_result

    except Exception as e:
        return InteractionExtractionResult(
            interactions=[],
            error_message=str(e),
        )

### Pre-Tagging


Description

In [4]:
df = pl.read_excel("../2026-06-08-test1.xlsx") # file with pmids + context blocks

In [5]:
from dgilit_wags.tagging import (
    BioBertEntityTagger,
    EntityPreTaggingService,
    SQLiteNormalizerCache,
    TaggerConfig,
    ViccNormalizer,
)

pretagger = EntityPreTaggingService(
    tagger=BioBertEntityTagger(
        TaggerConfig(
            batch_size=16,
            include_drugs=True,
            include_genes=True,
            include_diseases=False,
            device=-1,  # use 0 on CUDA
        )
    ),
    normalizer=ViccNormalizer(
        cache=SQLiteNormalizerCache(".dgilit_normalizer_cache.sqlite")
    ),
)

In [6]:
contexts = (
    df["context"]
    .fill_null("")
    .cast(pl.Utf8)
    .to_list()
)
pmids = df["pmid"].cast(str).to_list() if "pmid" in df.columns else [None] * len(contexts)
block_ids = [str(i) for i in range(len(contexts))]

tagged_blocks = pretagger.tag_blocks(
    contexts=contexts,
    pmids=pmids,
    block_ids=block_ids,
)

Device set to use cpu


Tagging text batches:   0%|          | 0/7 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def entity_display_name(e):
    if e.concept and e.concept.concept_label:
        return e.concept.concept_label
    if e.text:
        return e.text
    return None


def build_candidate_context(block):
    drugs = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "drug"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    genes = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "gene"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    return {
        "pmid": block.pmid,
        "drugs": drugs,
        "genes": genes,
        "context": block.context,
    }

candidate = build_candidate_context(tagged_blocks[0])
candidate

### LLM Calls


Description

In [ ]:
def entity_display_name(e):
    if e.concept and e.concept.concept_label and e.concept.concept_id:
        return e.concept.concept_label
    return None


def get_candidates(block):
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [entity_display_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [entity_display_name(e)]
        if name
    })

    return drugs, genes

In [ ]:
temperatures = [0]

def run_experiments(tagged_blocks, temperatures, num_runs, prompt_version: str):
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = InteractionExtractionPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for block in tqdm(
                tagged_blocks,
                total=len(tagged_blocks),
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                candidate_drugs, candidate_genes = get_candidates(block)

                # Huge cost saver: skip useless abstracts
                if not candidate_drugs or not candidate_genes:
                    results.append(
                        {
                            "pmid": block.pmid,
                            "block_id": block.block_id,
                            "skipped": True,
                            "skip_reason": "No normalized drug and gene candidates",
                            "interactions": [],
                        }
                    )
                    continue

                result = extraction_interactions(
                    task_runner=task_runner,
                    prompt=prompt,
                    context=block.context,
                    candidate_drugs=candidate_drugs,
                    candidate_genes=candidate_genes,
                )

                if result.error_message:
                    results.append(
                        {
                            "pmid": block.pmid,
                            "block_id": block.block_id,
                            "skipped": False,
                            "candidate_drugs": candidate_drugs,
                            "candidate_genes": candidate_genes,
                            "interactions": [],
                            "error_message": result.error_message,
                        }
                    )
                    continue

                results.append(
                    {
                        "pmid": block.pmid,
                        "block_id": block.block_id,
                        "skipped": False,
                        "candidate_drugs": candidate_drugs,
                        "candidate_genes": candidate_genes,
                        "interactions": [
                            interaction.model_dump()
                            for interaction in result.interactions
                        ],
                    }
                )

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs



In [ ]:
stored_runs = run_experiments(
    tagged_blocks,
    temperatures,
    num_runs=1,
    prompt_version="v1",
)

NOTE: After running this once, temperature as a parameter really doesn't need to be included. Kinda just set it to 0 and forget about it tbh.

In [ ]:
stored_runs = run_experiments(df, temperatures, num_runs=1, prompt_version='v1')

In [ ]:
stored_runs

### Save

Save the output from a wags-llm run. After running through this whole process but I think its good.

In [ ]:
rows = []
for run in stored_runs:
    for result in run["results"]:
        rows.append({
            "run_idx": run["run_idx"],
            "prompt_version": run["prompt_version"],
            "temperature": run["temperature"],
            "drug": result["drug"],
            "gene": result["gene"],
            "interaction": result["interaction"],
        })

df_results = pl.DataFrame(rows)

df_results.write_csv("2026-06-08-interaction_predictions.csv")
df_results.write_parquet("2026-06-08-interaction_predictions.parquet")

df_results.head()